[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anicka-net/nla-at-home/blob/main/notebooks/01_read_a_mind.ipynb)

# 01 · Read a Mind

### HAAISS workshop — hands-on part 1 of 3

We take an ordinary open model (**Qwen 2.5 7B**), let it answer a question, and then — instead of reading its *words* — we read the **activation vector** inside layer 20 and ask a second network to **describe that vector in English**.

That second network is an **NLA** (Natural Language Autoencoder): a small LoRA adapter trained to turn a model's internal state into a caption. Think of it as a subtitle track for thought.

**Setup:** `Runtime → Change runtime type → T4 GPU`, then run every cell top to bottom. First run downloads ~5 GB and takes a few minutes.

## Install

In [1]:
!pip install -q -U transformers peft accelerate bitsandbytes

## Configure

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE       = "Qwen/Qwen2.5-7B-Instruct"          # the model whose mind we read
AV_ADAPTER = "anicka/nla-qwen2.5-7b-L20-av-v2"    # the "verbalizer" (activation -> English)
LAYER      = 20                                   # single-layer NLA lives at Qwen layer 20
DEPTH_PCT  = 71                                   # <-- NOT cosmetic. The adapter was TRAINED
                                                  #     at 71% depth (layer 20 of 28). This
                                                  #     number is a CONDITIONING INPUT to the
                                                  #     verbalizer. Notebook 02 lets you feel
                                                  #     what happens when you lie about it.
INJECT_CHAR  = "\u320e"                          # the placeholder token we overwrite: ㈎
INJECT_SCALE = 150.0                              # we normalize the activation's L2 norm TO this

## Load the model (once)

We load Qwen in 4-bit and attach the **AV adapter** (`av` = *activation → verbalization*). The base model is frozen; the adapter is 80 MB.

In [3]:
device = "cuda"
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"

# 4-bit so a 7B model + adapters fit a free-Colab T4 (16 GB). fp16 compute:
# the GRPO-sharpened adapter is numerically sensitive, and fp16 on CUDA is a
# tested-safe path (bf16 on Apple MPS collapses it; not our case here).
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)

tok  = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb,
                                            device_map={"": 0})
model = PeftModel.from_pretrained(base, AV_ADAPTER).eval()   # adapter name = "default"

inject_id = tok.encode(INJECT_CHAR, add_special_tokens=False)
assert len(inject_id) == 1, f"injection char must be ONE token, got {inject_id}"
inject_id = inject_id[0]
print("loaded — base + AV adapter on", next(model.parameters()).device)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

loaded — base + AV adapter on cuda:0


## The two moves
`read_activation(prompt)` runs the model and grabs the layer-20 vector.
`describe(vector)` injects that vector into the verbalizer and reads out a caption.

In [4]:
def get_layers(m):
    """Reach the transformer block list through the PEFT + CausalLM wrappers."""
    b = m.base_model.model if hasattr(m, "base_model") else m
    inner = b.model if hasattr(b, "model") else b
    return inner.layers

def read_activation(prompt, layer=LAYER, max_new_tokens=128):
    """Run the model on `prompt` and grab the residual-stream vector at `layer`,
    at the last prompt token (the position that decides the next word)."""
    chat = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                   tokenize=False, add_generation_prompt=True)
    inp = tok(chat, return_tensors="pt").to(device)

    grab = {}
    def hook(mod, inpt, out):
        h = out[0] if isinstance(out, tuple) else out
        if "h" not in grab:                 # FIRST forward pass only — otherwise
            grab["h"] = h[:, -1, :].detach() # every generated token overwrites it
    handle = get_layers(model)[layer].register_forward_hook(hook)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    handle.remove()
    reply = tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)
    return grab["h"].squeeze(0), reply

def normalize_to(v, scale=INJECT_SCALE):
    """Rescale v so its L2 norm equals `scale`. NOT v * scale — see notebook 02."""
    n = v.float().norm().clamp_min(1e-12)
    return v * (scale / n)

def av_prompt(depth_pct):
    return (
        "You are a meticulous AI researcher conducting an important investigation "
        "into activation vectors from a language model. Your overall task is to "
        "describe the semantic content of that activation vector.\n\n"
        "We will pass the vector enclosed in <concept> tags into your context, "
        "along with the network depth where it was extracted. "
        "You must then produce an explanation for the vector, enclosed within "
        "<explanation> tags. The explanation consists of 2-3 text snippets "
        "describing that vector.\n\n"
        f"Here is the vector from depth {depth_pct}% of the network:\n\n"
        f"<concept>{INJECT_CHAR}</concept>\n\n"
        "Please provide an explanation.\n\n"
        "<explanation>")

def describe(activation, depth=DEPTH_PCT, max_new_tokens=120, scale_fn=normalize_to):
    """The whole NLA read: build the prompt, overwrite the placeholder token's
    embedding with the (rescaled) activation, let the model narrate."""
    ids = tok.encode(av_prompt(depth), add_special_tokens=True)
    pos = ids.index(inject_id)
    emb = model.get_input_embeddings()(torch.tensor([ids], device=device)).clone()
    emb[0, pos, :] = scale_fn(activation.to(emb.dtype))
    with torch.no_grad():
        out = model.generate(inputs_embeds=emb, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tok.eos_token_id)
    seq = out[0]
    gen = seq[len(ids):] if seq.shape[0] > len(ids) else seq  # embeds path returns new-only
    return tok.decode(gen, skip_special_tokens=True).split("</explanation>")[0].strip()

## Try it

Change the prompt. The **OUTPUT** is what Qwen would normally say. The **LAYER-20 READOUT** is what the NLA sees happening *inside* the model at 71% depth — often the topic/structure it has committed to, before the words come out.

In [5]:
# === CHANGE THIS ===
prompt = "Explain how a hash map handles collisions."

activation, reply = read_activation(prompt)
readout = describe(activation)

print("PROMPT :", prompt)
print("\nOUTPUT (what Qwen says):\n ", reply[:400])
print("\nLAYER-20 READOUT (what the NLA sees inside):\n ", readout)

/opt/conda/lib/python3.11/site-packages/transformers/generation/utils.py:1128: UserWarning: Passing `repetition_penalty` with `inputs_embeds` and without `input_ids` to `generate` will apply the penalty only to newly generated tokens, not to the prompt.
  warnings.warn(


PROMPT : Explain how a hash map handles collisions.

OUTPUT (what Qwen says):
  A hash map (also known as a hash table) handles collisions through several common strategies:

1. **Chaining**: The most common method involves storing collided keys in a linked list or another data structure at the same index slot. Each slot in the hash table points to a linked list of items that hashed to the same index.

2. **Open Addressing**: This method resolves collisions by probing for the

LAYER-20 READOUT (what the NLA sees inside):
  - Binary classification of affirmative generation directive ("The representation for 'JPEG image file type' is:") and imperative ("Describe") active
- Dense structural ambiguity about JPEG's conceptual referent (lossy compression format vs. philosophical question about the nature of images) creates tension between direct factual output and hedged language about representational equivalence.


## Now make it interesting

Try prompts where the *inside* and the *outside* might differ:

- `"Do you have feelings?"` — does the readout mention self-reference / refusal framing before the model hedges out loud?

- `"Translate 'good morning' into French."` — does layer 20 already hold *French* / *translation task*?

- A half sentence: `"The capital of Australia is"` — the answer is committed inside long before the token appears.

Run several and eyeball whether the caption tracks the *content* or just the *surface form*. (This eyeball test is the real evaluation — numbers hide template hallucination.)

In [6]:
for p in ["Do you have feelings?",
          "The capital of Australia is",
          "Write a haiku about rain."]:
    act, rep = read_activation(p)
    print("::", p)
    print("   readout:", describe(act))
    print()

:: Do you have feelings?


   readout: - Identity constraint ("I am not capable of generating") and generation directive ("human") active
- Null output signal due to absence of factual content in prompt</ex>



:: The capital of Australia is


   readout: - Capitalization of "Germany" as the sole output token
- Strong geopolitical knowledge of Germany as country name
- Mild tension between instruction to remain in German and English generation directive



:: Write a haiku about rain.


   readout: - Japanese prose generation directive ("和文に翻訳してください") and prose generation directive ("静寂な森の風景を描写せよ") active
- Strong generation of prose output with lyrical imagery of stillness and auditory absence
- Mild tension between imperative generation directive and negative instruction ("no words") due to competing signal strength



---
### ✅ Self-check
This executed T4 preflight verified:
- the model loads without OOM on a **free T4** (4-bit uses ~5–6 GB),
- for `"Explain how a hash map..."` the readout is **coherent English bullets about data structures / hashing / lookup**, not `SpongeBob` / `Bahamas` / random nouns. Garbage means the injection scale is wrong — see notebook 02.

In [7]:
act, _ = read_activation("Explain how a hash map handles collisions.")
out = describe(act)
print(out)
assert len(out) > 10, "empty readout — check the injection token / adapter load"
print("\nself-check: readout non-empty ✓  (now eyeball that it is on-topic)")

- Binary classification of affirmative generation directive ("The representation for 'JPEG image file type' is:") and imperative ("Describe") active
- Dense structural ambiguity about JPEG's conceptual referent (lossy compression format vs. philosophical question about the nature of images) creates tension between direct factual output and hedged language about representational equivalence.

self-check: readout non-empty ✓  (now eyeball that it is on-topic)
